In [1]:
import os
from pathlib import Path
from google.colab import drive

# Mount Google Drive to access the dataset
drive.mount('/content/drive')
print("✅ Google Drive mounted.")

# --- Configuration ---
# This path points to the root of your construction dataset
DATASET_ROOT = "/content/drive/MyDrive/Construction/dataset"

# 1. Define the content of data.yaml
# It includes the correct paths and the 10 class names.
yaml_content = f"""train: {DATASET_ROOT}/train/images
val: {DATASET_ROOT}/val/images
test: {DATASET_ROOT}/test/images

# Total number of classes for the Construction project
nc: 10
names: ['HardHat', 'SafetyVest', 'Person', 'Excavator', 'Ladder', 'Barricade', 'Tool', 'Vehicle', 'Gloves', 'SafetyGlasses']
"""

# 2. Define the path where data.yaml will be saved
DATA_YAML_PATH = Path(DATASET_ROOT) / 'data.yaml'

# 3. Write the content to the file
try:
    with open(DATA_YAML_PATH, 'w') as f:
        f.write(yaml_content)
    print(f"✅ data.yaml successfully created and saved at: {DATA_YAML_PATH}")

    # Display the content for confirmation
    print("\n--- Content of data.yaml ---")
    print(yaml_content)

except Exception as e:
    print(f"❌ An error occurred while writing data.yaml: {e}")

Mounted at /content/drive
✅ Google Drive mounted.
✅ data.yaml successfully created and saved at: /content/drive/MyDrive/Construction/dataset/data.yaml

--- Content of data.yaml ---
train: /content/drive/MyDrive/Construction/dataset/train/images
val: /content/drive/MyDrive/Construction/dataset/val/images
test: /content/drive/MyDrive/Construction/dataset/test/images

# Total number of classes for the Construction project
nc: 10
names: ['HardHat', 'SafetyVest', 'Person', 'Excavator', 'Ladder', 'Barricade', 'Tool', 'Vehicle', 'Gloves', 'SafetyGlasses']



In [2]:
# Install Ultralytics library
!pip install ultralytics --quiet
print("✅ Ultralytics installed.")

# Mount Google Drive to access the dataset
from google.colab import drive
from ultralytics import YOLO
import os

# Ensure Drive is mounted
drive.mount('/content/drive')
print("✅ Google Drive mounted.")

# --- CONFIGURATION ---
# Define the root directory based on the Construction Detection project path
DATASET_ROOT = "/content/drive/MyDrive/Construction/dataset"

# Define the path to the CORRECT data.yaml file (created in the previous step)
# This path points to the construction dataset configuration, NOT the smoke detection one.
DATA_YAML_PATH = os.path.join(DATASET_ROOT, "data.yaml")

print(f"\n✅ Training configuration loaded from: {DATA_YAML_PATH}")


# The section below was removed as the data.yaml was successfully created/confirmed
# and the old content was for a different, smoke detection project:
#
# yaml_content = """train: /content/drive/MyDrive/Smoke Detection/datasets/train/images ... """
# DATA_YAML_PATH = "/content/drive/MyDrive/Smoke Detection/datasets/data.yaml"
# with open(DATA_YAML_PATH, "w") as f: f.write(yaml_content)
# print("✅ data.yaml confirmed.")


# 4. Load YOLOv8 Model
# Start with the efficient nano model
model = YOLO("yolov8n.pt")
print("✅ YOLOv8n model loaded.")


# 5. Train the Model for Construction Detection
print("\n🚀 Starting training for Construction Detection...")

results = model.train(
    data=DATA_YAML_PATH, # Uses the correct Construction data.yaml path
    epochs=50,
    imgsz=640,
    batch=16,
    # Updated name to reflect the Construction Detection task
    name="construction_yolov8n_v1",
    device=0, # Use GPU (Colab T4)
    workers=4 # Use more workers for faster data loading
)

print("\n✅ Training initiated. The results will be saved in the 'runs' folder within your Google Drive.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 71.2 MB/s eta 0:00:00
✅ Ultralytics installed.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted.

✅ Training configuration loaded from: /content/drive/MyDrive/Construction/dataset/data.yaml
✅ YOLOv8n model loaded.

🚀 Starting training for Construction Detection...
Ultralytics 8.3.228 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compi

In [1]:
import os
from ultralytics import YOLO
from google.colab import files
from IPython.display import Video, display

# --- CONFIGURATION (Must match the previous training run) ---
# Define the run name used during training
TRAIN_RUN_NAME = "construction_yolov8n_v1"
RUNS_PATH = f"/content/runs/detect/{TRAIN_RUN_NAME}"
BEST_WEIGHTS_PATH = os.path.join(RUNS_PATH, "weights", "best.pt")

# Define the source video path.
# IMPORTANT: You must upload your video (e.g., 'indianworkers.mp4')
# to the current Colab session's root folder (/content/) before running this.
SOURCE_VIDEO_PATH = "/content/indianworkers.mp4"

# Define the output directory to save the annotated video (in your Drive)
OUTPUT_DIR = "/content/drive/MyDrive/Construction/video_predictions"
# -------------------------------------------------------------

# 1. Load the Best Weights
try:
    print(f"🧠 Loading best weights from: {BEST_WEIGHTS_PATH}")
    model = YOLO(BEST_WEIGHTS_PATH)
    print("✅ Best model weights loaded successfully.")

except Exception as e:
    print(f"❌ Error loading best weights. Please ensure the file exists at: {BEST_WEIGHTS_PATH}. Error: {e}")
    exit()

# 2. Check if the source video exists (Skipped, assuming the user has confirmed the path)
if not os.path.exists(SOURCE_VIDEO_PATH):
    print(f"\n⚠️ WARNING: Source video not found at '{SOURCE_VIDEO_PATH}'.")
    print("Please upload your test video (e.g., 'indianworkers.mp4') to the Colab environment using the file explorer on the left or the 'files.upload()' method.")
    exit()

# 3. Run Predictions on the Video
print(f"\n🖼️ Running predictions on the video: {SOURCE_VIDEO_PATH}")
results = model.predict(
    source=SOURCE_VIDEO_PATH,
    save=True, # Saves the annotated video
    project=OUTPUT_DIR, # Saves results in this directory
    name=TRAIN_RUN_NAME, # Creates a subdirectory under OUTPUT_DIR
    conf=0.25 # Adjust confidence threshold if needed
)

# 4. Get the path to the saved video
# The prediction output path can be found in the results object
saved_video_path = results[0].save_dir # Get the directory where the video was saved
annotated_video_file = os.path.join(saved_video_path, os.path.basename(SOURCE_VIDEO_PATH))

print(f"\n✅ Video prediction complete. Annotated video saved to Drive at:\n**{annotated_video_file}**")
print("\n--- Displaying Annotated Video (requires a few moments to save and load) ---")

# 5. Display the annotated video in the notebook
try:
    # Display the final saved video for immediate review (if small enough)
    display(Video(annotated_video_file, embed=True, html_attributes='controls loop autoplay muted'))
except Exception as e:
    print(f"❌ Could not display video in notebook. Check the file manually at the path above. Error: {e}")

🧠 Loading best weights from: /content/runs/detect/construction_yolov8n_v1/weights/best.pt
✅ Best model weights loaded successfully.

🖼️ Running predictions on the video: /content/indianworkers.mp4

WARNING ⚠️ 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/648) /content/indianworkers.mp4: 384x640 2 HardHats, 1 Ladder, 2 Barricades, 2 Vehicles, 3 Glovess, 1 SafetyGlasses, 43.1ms
video 1/1 (frame 2/648) /content/indianworkers.mp4: 384x640 1 HardHat, 1 Ladder, 2 Barricades, 2 Vehicles, 3 Glovess

In [4]:
from ultralytics import YOLO
import os

# --- CONFIGURATION (Must match the previous training run) ---
# Define the run name used during training (this is where your weights are saved)
TRAIN_RUN_NAME = "construction_yolov8n_v1"
RUNS_PATH = f"/content/runs/detect/{TRAIN_RUN_NAME}"
BEST_WEIGHTS_PATH = os.path.join(RUNS_PATH, "weights", "best.pt")

# The data.yaml used during training implicitly defines the validation dataset location.
# -------------------------------------------------------------

# 1. Load the Best Weights
try:
    print(f"🧠 Loading best weights from: {BEST_WEIGHTS_PATH}")
    # Load the model with the best weights from the completed training run
    model = YOLO(BEST_WEIGHTS_PATH)
    print("✅ Best model weights loaded successfully.")

except Exception as e:
    print(f"❌ Error loading best weights. Please ensure the training completed successfully and the file exists at: {BEST_WEIGHTS_PATH}. Error: {e}")
    # Exit if model load fails to prevent subsequent errors
    exit()

# 2. Run Validation
print("\n🔍 Running final validation on best weights...")
print("This evaluates the model's performance (mAP, F1, etc.) on the validation set.")

# The model.val() method will automatically run on the validation split defined
# in the data.yaml file used during the original training run.
results = model.val()

print("\n--- Validation Results Summary ---")
print(f"  mAP50-95: {results.box.map}")
print(f"  mAP50:    {results.box.map50}")
print(f"  mAP75:    {results.box.map75}")

🧠 Loading best weights from: /content/runs/detect/construction_yolov8n_v1/weights/best.pt
✅ Best model weights loaded successfully.

🔍 Running final validation on best weights...
This evaluates the model's performance (mAP, F1, etc.) on the validation set.
Ultralytics 8.3.228 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,007,598 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 3.0±1.1 MB/s, size: 45.9 KB)
val: Scanning /content/drive/MyDrive/Construction/dataset/val/labels.cache... 114 images, 10 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 114/114 129.7Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 2.1it/s 3.8s
                   all        114        697       0.85      0.718      0.786      0.472
               HardHat         42         79      0.913      0.747      0.867      0.554
            SafetyVest         19      

In [7]:
import time
import statistics
import random

# --- Configuration for Simulation ---
# Simulating the video parameters
TOTAL_FRAMES = 648
VIDEO_FPS = 30 # Original video capture rate (often 25, 30, or 60)
MODEL_NAME = "YOLOv8n-v1"

# Simulating the latency observed in your logs (7.0 ms to 15.0 ms)
# This represents the model's total time (preprocess + inference + postprocess)
MIN_SIMULATED_LATENCY_MS = 7.0
MAX_SIMULATED_LATENCY_MS = 15.0

# Store the latency measurements for analysis
frame_latencies_ms = []

def simulate_detection_process(frame_number):
    """
    Simulates the actual object detection process for a single video frame.

    In a real application, this function would contain:
    1. Preprocessing (resizing, normalization).
    2. Model inference (the forward pass).
    3. Postprocessing (NMS, drawing boxes).
    """
    # Simulate variable processing time based on the model's performance
    # Convert milliseconds to seconds for time.sleep()
    latency_ms = random.uniform(MIN_SIMULATED_LATENCY_MS, MAX_SIMULATED_LATENCY_MS)
    latency_s = latency_ms / 1000.0

    # Simulate the work being done by the detection model
    time.sleep(latency_s)

    # Return the measured latency to mimic a real detection library's output
    return latency_ms

def process_video_frames():
    """Iterates through all frames and calculates speed metrics."""
    print(f"🧠 Starting video analysis simulation ({TOTAL_FRAMES} frames)...")

    start_time_total = time.time() # Start global timer

    for i in range(1, TOTAL_FRAMES + 1):
        # 1. Start timer for the current frame
        frame_start_time = time.time()

        # 2. Run the detection (simulated)
        # In a real app: results = model(frame)
        latency_ms = simulate_detection_process(i)

        # 3. End timer for the current frame
        frame_end_time = time.time()

        # 4. Calculate the *total* time taken for this frame (including overhead)
        # This is equivalent to the number you saw in your logs (e.g., 8.5ms)
        total_frame_time_ms = (frame_end_time - frame_start_time) * 1000
        frame_latencies_ms.append(total_frame_time_ms)

        # Print the log output similar to what you observed
        # Note: The simulated_latency is close to but not identical to the total_frame_time due to Python/OS overhead.
        if i < 20 or i % 50 == 0: # Print a few frames and then every 50th for brevity
            print(f"video 1/1 (frame {i}/{TOTAL_FRAMES}): Detection finished, total time {total_frame_time_ms:.1f}ms")

    end_time_total = time.time()

    # --- Final Calculation and Summary ---

    total_processing_time = end_time_total - start_time_total

    # Calculate Averages
    avg_latency_ms = statistics.mean(frame_latencies_ms)

    # Calculate FPS (Frames per Second)
    # FPS = 1000 / Average Latency (ms)
    avg_fps = 1000 / avg_latency_ms

    print("\n" + "="*50)
    print("           ✨ FPS AND LATENCY REPORT ✨")
    print("="*50)
    print(f"Total Frames Processed: {TOTAL_FRAMES}")
    print(f"Total Processing Time: {total_processing_time:.2f} seconds")
    print("-" * 50)
    print(f"Average Per-Frame Latency: {avg_latency_ms:.2f} ms")
    print(f"Frames Per Second (FPS): {avg_fps:.2f} FPS")

    # Compare with the original video's FPS
    if avg_fps > VIDEO_FPS:
        print(f"\n✅ Model runs faster than the video's native {VIDEO_FPS} FPS.")
    else:
        print(f"\n⚠️ Model runs slower than the video's native {VIDEO_FPS} FPS.")


if __name__ == "__main__":
    process_video_frames()

🧠 Starting video analysis simulation (648 frames)...
video 1/1 (frame 1/648): Detection finished, total time 16.6ms
video 1/1 (frame 2/648): Detection finished, total time 14.6ms
video 1/1 (frame 3/648): Detection finished, total time 8.6ms
video 1/1 (frame 4/648): Detection finished, total time 9.1ms
video 1/1 (frame 5/648): Detection finished, total time 13.8ms
video 1/1 (frame 6/648): Detection finished, total time 10.3ms
video 1/1 (frame 7/648): Detection finished, total time 10.4ms
video 1/1 (frame 8/648): Detection finished, total time 10.8ms
video 1/1 (frame 9/648): Detection finished, total time 8.2ms
video 1/1 (frame 10/648): Detection finished, total time 10.1ms
video 1/1 (frame 11/648): Detection finished, total time 12.9ms
video 1/1 (frame 12/648): Detection finished, total time 15.0ms
video 1/1 (frame 13/648): Detection finished, total time 10.9ms
video 1/1 (frame 14/648): Detection finished, total time 8.5ms
video 1/1 (frame 15/648): Detection finished, total time 14.9ms
